In [ ]:
%%bash
pip install jupyterplot
pip install pyserial
pip install git+https://github.com/lbaitemple/Proto-Grid.git@arnabEnhancements
pip install jupyterlab-executor

In [ ]:
%%bash
pip install bokeh --upgrade
pip show revidyne

In [ ]:
!mkdir -p streaming

In [ ]:
%%writefile streaming/main.py

from revidyne import AllDevice
from bokeh import models, plotting, io
from bokeh.io import show, output_notebook
from bokeh.layouts import column, row
import pandas as pd
from itertools import cycle
from datetime import datetime
import time, math, sys, re, os
from contextlib import contextmanager
import io as console_io
import numpy as np
import heapq

a=AllDevice(115200)
mm=a.getAllDevice()
time.sleep(8)
for key in mm.keys():
    mm[key].getDevice().call('init')
    """
    time.sleep(0.2)
    if key.startswith('fan'):
        mm[key].getDevice().call('fanOff')
    elif key.startswith('generator'):
        mm[key].getDevice().call('trackOff')
    elif key.startswith('houseload'):
        mm[key].getDevice().call('lightsOut') 
    elif key.startswith('solartracker'):
        mm[key].getDevice().call('trackOff')
    elif key.startswith('windturbine'):
        mm[key].getDevice().call('trackOff')
    """
    time.sleep(0.2)
time.sleep(8)
        
i = 0
c = 0
outKWGens = []
outKWSols = []
outKWWTs = []
outVGens = []
outVSols = []
outVWTs = []
outFreqGens = []
co2accum = None
demand = None
surplus = None
needle = None
window_universal = 120
dial_min = -1500
dial_max = 1500
gen_max = 250
pq_map = set()
pq_demand = []
pq_supply = []
power_demand = 0
power_supply = 0

@contextmanager
def suppress_stdout():
    with open(os.devnull, "w") as devnull:
        old_stdout = sys.stdout
        sys.stdout = devnull
        try:  
            yield
        finally:
            sys.stdout = old_stdout

def update(event=None):
    global i, c, outKWGens, outKWSols, outKWWTs, co2accum, outVGens, outVSols, outVWTs, outFreqGens
    global demand, surplus, needle, power_demand, power_supply
    i+=1
    totalSupply = 0
    totalDemand = 0
    
    for j, gen in enumerate([x for x in mm.keys() if x.startswith('generator')]):
        with suppress_stdout():
            outLoad = 0
            try:
                outLoad = float(mm[gen].getDevice().call('getKW')[0])
            except:
                pass
        outLoad = 0 if math.isnan(outLoad) else outLoad
        totalSupply += outLoad
        new_data={'Time': [i], 'OutputLoad': [outLoad]}
        outKWGens[j].stream(new_data, rollover=window_universal)
        with suppress_stdout():
            outVolt = 0
            try:
                outVolt = float(mm[gen].getDevice().call('getVal')[0])
            except:
                pass
        outVolt = 0 if math.isnan(outVolt) else outVolt
        new_data={'Time': [i], 'VoltageOutput': [outVolt]}
        outVGens[j].stream(new_data, rollover=window_universal)
        with suppress_stdout():
            outFreq = 0
            try:
                outFreq = float(mm[gen].getDevice().call('getFreq')[0])
            except:
                pass
        outFreq = 0 if math.isnan(outFreq) else outFreq
        new_data={'Time': [i], 'Frequency': [outFreq]}
        outFreqGens[j].stream(new_data, rollover=window_universal)
        with suppress_stdout():
            outCO2 = 0
            try:
                outCO2 = float(mm[gen].getDevice().call('getCarbon')[0])
            except:
                pass
        outCO2 = 0 if math.isnan(outCO2) else outCO2
        c += outCO2

    for j, sol in enumerate([x for x in mm.keys() if x.startswith('solar')]):
        with suppress_stdout():
            outLoad = 0
            try:
                outLoad = float(mm[sol].getDevice().call('getKW')[0])
            except:
                pass
        outLoad = 0 if math.isnan(outLoad) else outLoad
        totalSupply += outLoad
        new_data={'Time': [i], 'OutputLoad': [outLoad]}
        outKWSols[j].stream(new_data, rollover=window_universal)
        with suppress_stdout():
            outVolt = 0
            try:
                outVolt = float(mm[sol].getDevice().call('getVal')[0])
            except:
                pass
        outVolt = 0 if math.isnan(outVolt) else outVolt
        new_data={'Time': [i], 'VoltageOutput': [outVolt]}
        outVSols[j].stream(new_data, rollover=window_universal)
        with suppress_stdout():
            outFreq = 0
            try:
                outFreq = float(mm[sol].getDevice().call('getFreq')[0])
            except:
                pass
        outFreq = 0 if math.isnan(outFreq) else outFreq
        new_data={'Time': [i], 'Frequency': [outFreq]}
        outFreqGens[j].stream(new_data, rollover=window_universal)
        with suppress_stdout():
            outCO2 = 0
            try:
                outCO2 = float(mm[sol].getDevice().call('getCarbon')[0])
            except:
                pass
        outCO2 = 0 if math.isnan(outCO2) else outCO2
        c += outCO2

    for j, wind in enumerate([x for x in mm.keys() if x.startswith('wind')]):
        with suppress_stdout():
            outLoad = 0
            try:
                outLoad = float(mm[wind].getDevice().call('getKW')[0])
            except:
                pass
        outLoad = 0 if math.isnan(outLoad) else outLoad
        totalSupply += outLoad
        new_data={'Time': [i], 'OutputLoad': [outLoad]}
        outKWWTs[j].stream(new_data, rollover=window_universal)
        with suppress_stdout():
            outVolt = 0
            try:
                outVolt = float(mm[wind].getDevice().call('getVal')[0])
            except:
                pass
        outVolt = 0 if math.isnan(outVolt) else outVolt
        new_data={'Time': [i], 'VoltageOutput': [outVolt]}
        outVWTs[j].stream(new_data, rollover=window_universal)
        with suppress_stdout():
            outFreq = 0
            try:
                outFreq = float(mm[wind].getDevice().call('getFreq')[0])
            except:
                pass
        outFreq = 0 if math.isnan(outFreq) else outFreq
        new_data={'Time': [i], 'Frequency': [outFreq]}
        outFreqGens[j].stream(new_data, rollover=window_universal)
        with suppress_stdout():
            outCO2 = 0
            try:
                outCO2 = float(mm[wind].getDevice().call('getCarbon')[0])
            except:
                pass
        outCO2 = 0 if math.isnan(outCO2) else outCO2
        c += outCO2

    for j, house in enumerate([x for x in mm.keys() if x.startswith('houseload')]):
        outLoad = household_get_call(mm[house].getDevice(), 'getLoadVal')
        totalDemand += outLoad

    new_data={'Time': [i], 'HouseLoads': [totalDemand]}
    demand.stream(new_data, rollover=window_universal)
    new_data={'Time': [i], 'CO2Accumulation': [c]}
    co2accum.stream(new_data, rollover=window_universal)
    power_demand = totalDemand
    power_supply = totalSupply
    s = totalSupply - totalDemand
    current_data_surplus = dict(surplus.data)
    current_data_surplus['labels']=[str("{:.1f}".format(s))]
    surplus.data = current_data_surplus
    current_data_needle = dict(needle.data)
    needle_angle = np.pi - ((s - dial_min) / (dial_max - dial_min)) * np.pi
    x = 0.8 * np.cos(needle_angle)
    y = 0.8 * np.sin(needle_angle)
    current_data_needle['x']=[x]
    current_data_needle['y']=[y]
    current_data_needle['angle']=[needle_angle]
    needle.data = current_data_needle
    

def create_dial():
    global surplus, needle
    z = plotting.figure(width=300, height=300, title="Power Surplus (kW)")
    z.toolbar_location = None
    z.x_range = models.Range1d(start=-1.2, end=1.2)
    z.y_range = models.Range1d(start=-0.4, end=1.2)
    z.axis.visible = False
    z.grid.visible = False
    z.annular_wedge(x = 0, y = 0, inner_radius = 0.8, outer_radius = 1, start_angle=2 * np.pi / 3, end_angle=np.pi, fill_color="red")
    z.annular_wedge(x = 0, y = 0, inner_radius = 0.8, outer_radius = 1, start_angle=np.pi / 3, end_angle=2 * np.pi / 3, fill_color="orange")
    z.annular_wedge(x = 0, y = 0, inner_radius = 0.8, outer_radius = 1, start_angle=0, end_angle=np.pi / 3, fill_color="lightgreen")
    
    needle = models.ColumnDataSource(data=dict(x=[0], y=[0.8], angle=[0]))
    z.add_layout(models.Arrow(x_start=0, y_start=0, x_end="x", y_end="y", line_width=2, line_color="black", 
                            source=needle, end=models.VeeHead(size=5)))
    
    surplus = models.ColumnDataSource(dict(x=[0], y=[-0.1], labels=[str(0)]))
    z.add_layout(models.LabelSet(x='x', y='y', text='labels', source=surplus, text_align='center', 
                            text_font_size='10pt', level='glyph'))
    return z

def plotGenerators():
    global i, outKWGens, outVGens
    colors = ["red", "blue", "green", "pink", "brown"]
    for j, gen in enumerate([x for x in mm.keys() if x.startswith('generator')]):
        outKWGens.append(models.ColumnDataSource(data=dict(Time=[i], OutputLoad=[0])))
        outVGens.append(models.ColumnDataSource(data=dict(Time=[i], VoltageOutput=[0])))
        p.line(x="Time", y="OutputLoad", source=outKWGens[j], width=2, color=colors[j], legend_label=gen)
        pv.line(x="Time", y="VoltageOutput", source=outVGens[j], width=2, color=colors[j], legend_label=gen)

def plotSolars():
    global i, outKWSols, outVSols
    colors = ["red", "blue", "green", "pink", "brown"]
    for j, gen in enumerate([x for x in mm.keys() if x.startswith('solar')]):
        outKWSols.append(models.ColumnDataSource(data=dict(Time=[i], OutputLoad=[0])))
        outVSols.append(models.ColumnDataSource(data=dict(Time=[i], VoltageOutput=[0])))
        q.line(x="Time", y="OutputLoad", source=outKWSols[j], width=2, color=colors[j], legend_label=gen)
        qv.line(x="Time", y="VoltageOutput", source=outVSols[j], width=2, color=colors[j], legend_label=gen)

def plotWindTurbines():
    global i, outKWWTs, outVWTs
    colors = ["red", "blue", "green", "pink", "brown"]
    for j, gen in enumerate([x for x in mm.keys() if x.startswith('wind')]):
        outKWWTs.append(models.ColumnDataSource(data=dict(Time=[i], OutputLoad=[0])))
        outVWTs.append(models.ColumnDataSource(data=dict(Time=[i], VoltageOutput=[0])))
        r.line(x="Time", y="OutputLoad", source=outKWWTs[j], width=2, color=colors[j], legend_label=gen)
        rv.line(x="Time", y="VoltageOutput", source=outVWTs[j], width=2, color=colors[j], legend_label=gen)

def plotDemandAndCO2():
    global i, demand, co2accum
    demand = models.ColumnDataSource(data=dict(Time=[i], HouseLoads=[0]))
    co2accum = models.ColumnDataSource(data=dict(Time=[i], CO2Accumulation=[0]))
    
    s.line(x="Time", y="HouseLoads", source=demand, width=2)
    t.line(x="Time", y="CO2Accumulation", source=co2accum, width=2)

"""
def plotGauge():
    global i, surplus
    surplus = models.ColumnDataSource(data=dict(Time=[i], PowerSurplus=[0]))
    gauge = models.Gauge(x=200, y=200, radius=150, min_value=0, max_value=1000, value=surplus.data.PowerSurplus[0], title="Surplus Power (kW)")
    u.add_layout(gauge)
"""

def household_get_call(dev, cmd):
    output_capture = console_io.StringIO()
    original_stdout = sys.stdout
    sys.stdout = output_capture
    try:
        dev.call(cmd)
    except:
        pass
    finally:
        sys.stdout = original_stdout
    captured_output = 0.0
    try:
        captured_output = float(re.sub("[^0-9\.]", "", output_capture.getvalue().split(':')[1]))
    except:
        captured_output = 0.0
    output_capture.close()
    powerOut = captured_output
    if powerOut != 0:
        powerOut = 200 - powerOut
    return powerOut

def button_handler_apply(key1, cmd1, val1, but1):
    def apply_handler():
        dev = mm[key1].getDevice()
        plain_text = re.sub(r'<.*?>', '', cmd1.text)
        if plain_text.startswith('set'):
            new_row = {'Device': [key1], 'Command': [plain_text], 'Value': [val1.value]}
            cmds.stream(new_row, rollover=window_universal)
            if plain_text == 'setLoad':
                getattr(dev, plain_text)(int(val1.value))
            else:
                getattr(dev, plain_text)(float(val1.value))
        elif plain_text.startswith('get'):
            getVal = 0
            if key1.startswith('houseload'):
                getVal = household_get_call(dev, plain_text)
            else:
                with suppress_stdout():
                    getVal = dev.call(plain_text)[0]
            new_row = {'Device': [key1], 'Command': [plain_text], 'Value': [getVal]}
            cmds.stream(new_row, rollover=window_universal)
        elif plain_text != "":
            dev.call(plain_text)
            new_row = {'Device': [key1], 'Command': [plain_text], 'Value': [""]}
            cmds.stream(new_row, rollover=window_universal)    
        but1.disabled=True
    return apply_handler

"""
def button_handler_control(but1):
    def control_handler():
        pass
    return control_handler
"""

def dropdown_handler_selectCmd(cmd1, but1):
    def selectCmd_handler(event):
        cmd1.text = "<b>" + str(event.item) + "</b>"
        but1.disabled=False
    return selectCmd_handler

def button_handler_set_priority(key,priority,setPriority):
    def set_priority():
        p = 0
        try:
            if int(priority.value) >= 1 and int(priority.value) <= 5:
                p = -int(priority.value)
        except:
            pass
        if key not in pq_map:
            pq_map.add(key)
            if key.startswith('houseload'):
                heapq.heappush(pq_demand, (p, key))
            else:
                heapq.heappush(pq_supply, (p, key))
            print ("After setting priority queues:")
            print ("Demand :", pq_demand)
            print ("Supply :", pq_supply)
    return set_priority
        
widgets=[]

cmds = models.ColumnDataSource(data=dict(Device=[], Command=[], Value=[]))
columns = [
    models.TableColumn(field='Device', title='Device'),
    models.TableColumn(field='Command', title='Command'),
    models.TableColumn(field='Value', title='Value')
]
cmdString = ""
cmdCount = 0
data_table = models.DataTable(source=cmds, columns=columns, width=300, fit_columns=True)

for key in sorted(mm.keys()):
    if key.startswith('fan'):
        continue
    label = re.sub(r'\d+', '', key).upper()
    text = models.Div(text="<b>" + label + "</b>", width=100, height=30)
    cmdLst = [k for k in mm[key].getDevice().list_cmds().keys() if k != 'eoc' and not k.startswith('fan')]
    dropdown = models.Dropdown(width=150, height=30, label=key, menu=[(j, j) for j in cmdLst])
    inputCmd = models.Div(text="", width=100, height=30)
    inputVal = models.TextInput(width=100, height=30)
    button = models.widgets.Button(label="Apply", width=50, height=30)
    priority = models.TextInput(width=100, height=30)
    setPriority = models.widgets.Button(label="Priority(1-5)", width=80, height=30)
    button.disabled=True
    # button1 = models.widgets.Button(label="Control", width=50, height=30)
    
    dropdown.on_click(dropdown_handler_selectCmd(inputCmd, button))
    button.on_click(button_handler_apply(str(dropdown.label),inputCmd,inputVal,button))
    # button1.on_click(button_handler_control(button1))
    setPriority.on_click(button_handler_set_priority(key,priority,setPriority))
    
    widgets.append(row(text, dropdown, inputCmd, inputVal, button, priority, setPriority))
 
dropdown_layout = column(*widgets)
data_table.height = max(100, len(mm.keys()) * 40)

spacer_width = models.Spacer(width=70)
spacer_height = models.Spacer(height=40)

def setWindow (attr, old, new):
    window_universal = new

slide_window = models.widgets.Slider(start=20, end=500, value=80, step=20, title="Sliding Window Time Range")
slide_window.on_change('value', setWindow)

def setFans (attr, old, new):
    fanSpeed = new
    oldfanSpeed = old
    for j, fan in enumerate([x for x in mm.keys() if x.startswith('fan')]):
        with suppress_stdout():
            try:
                if fanSpeed == 0:
                    mm[fan].getDevice().call('fanOff')
                elif oldfanSpeed == 0:
                    mm[fan].getDevice().call('fanOn')
                    getattr(mm[fan].getDevice(), 'setSpeed')(int(fanSpeed))
                else:
                    getattr(mm[fan].getDevice(), 'setSpeed')(int(fanSpeed))
            except:
                pass

slide_fan = models.widgets.Slider(start=0, end=200, value=10, step=10, title="Fans Control")
slide_fan.on_change('value', setFans)

def button_stop_handler(buttonStop):
    def stop_handler():
        global pq_demand, pq_supply, pq_map
        for j, gen in enumerate(mm.keys()):
            if gen.startswith('generator') or gen.startswith('solar') or gen.startswith('wind'):
                mm[gen].getDevice().call("trackOff")
            elif gen.startswith('fan'):
                mm[gen].getDevice().call("fanOff")
            else:
                mm[gen].getDevice().call("off") 
        pq_demand = []
        pq_supply = []
        pq_map.clear()
    return stop_handler

buttonStop = models.widgets.Button(label="Stop", width=50, height=30)
buttonStop.on_click(button_stop_handler(buttonStop))

def button_bal_supply_handler(buttonBalSupply):
    def bal_supply_handler():
        global pq_demand, pq_supply, pq_map
        global power_demand
        print("Balancing Supply")
        temp = []
        prev = -6
        total = 0
        heapq.heappush(pq_supply,(1,'generator_eof'))
        while (pq_supply):
            q = heapq.heappop(pq_supply)
            dev_pri = q[0]
            dev_name = q[1]
            print("Acting on balancing for", dev_name)
            if not dev_name.startswith('generator'):
                with suppress_stdout():
                    outLoad = 0
                    try:
                        outLoad = float(mm[dev_name].getDevice().call('getKW')[0])
                    except:
                        pass
                    outLoad = 0 if math.isnan(outLoad) else outLoad
                total += outLoad
                pq_map.remove(dev_name)
            else:
                if dev_pri != prev:
                    j = 0
                    for dev in temp:
                        print("Balancing for", dev, "with demand", power_demand, "and current power total", total, "length", len(temp))
                        if total < power_demand:
                            mm[dev].getDevice().call('trackOn')
                            time.sleep(0.2)
                            getattr(mm[dev].getDevice(), 'setLoad')(min((int(power_demand) - int(total))//len(temp), gen_max))
                            print("Setting load output of", dev, "to", str(min((int(power_demand) - int(total))//len(temp), gen_max)))
                            j += min((int(power_demand) - int(total))//len(temp), gen_max)
                        else:
                            mm[dev].getDevice().call('trackOff')
                        pq_map.remove(dev)
                    total += j
                    temp = []
                prev = dev_pri
                temp.append(dev_name)
            
    return bal_supply_handler
    
def button_bal_demand_handler(buttonBalDemand):
    def bal_demand_handler():
        global pq_demand, pq_supply, pq_map
        global power_supply
        total_consumption = 0
        print("Balancing Demand")
        while (pq_demand):
            q = heapq.heappop(pq_demand)
            dev_pri = q[0]
            dev_name = q[1]
            pq_map.remove(dev_name)
            outLoad = household_get_call(mm[dev_name].getDevice(), 'getLoadVal')
            total_consumption += outLoad
            if total_consumption > power_supply:
                mm[dev_name].getDevice().call('lightsOut')
                
    return bal_demand_handler

buttonBalSupply = models.widgets.Button(label="Balance Supply", width=120, height=30)
buttonBalSupply.on_click(button_bal_supply_handler(buttonBalSupply))

buttonBalDemand = models.widgets.Button(label="Balance Demand", width=120, height=30)
buttonBalDemand.on_click(button_bal_demand_handler(buttonBalDemand))

layout_with_tab = row(dropdown_layout, spacer_width, data_table)
io.curdoc().add_root(layout_with_tab)

io.curdoc().add_root(row(slide_window, buttonStop, spacer_width, buttonBalSupply, buttonBalDemand))
io.curdoc().add_root(slide_fan)

p = plotting.figure(
    x_axis_label="Time", y_axis_label="OutputLoad", title="Generator(s) Output Load (kW)",
    width=550, height=300, x_axis_type="linear",
)
p.add_layout(models.Legend(), 'right')
q = plotting.figure(
    x_axis_label="Time", y_axis_label="OutputLoad", title="Solar Tracker(s) Output Load (kW)",
    width=550, height=300, x_axis_type="linear",
)
q.add_layout(models.Legend(), 'right')
r = plotting.figure(
    x_axis_label="Time", y_axis_label="OutputLoad", title="Wind Turbine(s) Output Load (kW)",
    width=550, height=300, x_axis_type="linear",
)
r.add_layout(models.Legend(), 'right')

pv = plotting.figure(
    x_axis_label="Time", y_axis_label="VoltageOutput", title="Generator(s) Output Voltage (V)",
    width=550, height=300, x_axis_type="linear",
)
pv.add_layout(models.Legend(), 'right')
qv = plotting.figure(
    x_axis_label="Time", y_axis_label="VoltageOutput", title="Solar Tracker(s) Output Voltage (V)",
    width=550, height=300, x_axis_type="linear",
)
qv.add_layout(models.Legend(), 'right')
rv = plotting.figure(
    x_axis_label="Time", y_axis_label="VoltageOutput", title="Wind Turbine(s) Output Voltage (V)",
    width=550, height=300, x_axis_type="linear",
)
rv.add_layout(models.Legend(), 'right')

s = plotting.figure(
    x_axis_label="Time", y_axis_label="HouseLoads", title="Power Demand (kW)",
    width=400, height=300, x_axis_type="linear",
)
t = plotting.figure(
    x_axis_label="Time", y_axis_label="CO2Accumulation", title="Cumulative CO2 Aggregated (Tons)",
    width=400, height=300, x_axis_type="linear",
)
u = create_dial()

plotGenerators()
plotSolars()
plotWindTurbines()
plotDemandAndCO2()
# plotGauge()

io.curdoc().add_root(row(p, q, r))
io.curdoc().add_root(row(pv, qv, rv))
io.curdoc().add_root(row(spacer_width, spacer_width, spacer_width, s, t, u))
io.curdoc().add_periodic_callback(update, 2000)

def plotGeneratorEfficiency():
    global i
    # Create a new ColumnDataSource for generator efficiency
    outEfficiencyGens = []
    colors = ["red", "blue", "green", "pink", "brown"]
    for j, gen in enumerate([x for x in mm.keys() if x.startswith('generator')]):
        outEfficiencyGens.append(models.ColumnDataSource(data=dict(Time=[i], Efficiency=[0])))
        peff.line(x="Time", y="Efficiency", source=outEfficiencyGens[j], width=2, color=colors[j], legend_label=f"{gen} Efficiency")

# Create a new figure for generator efficiency
peff = plotting.figure(
    x_axis_label="Time", y_axis_label="Efficiency", title="Generator(s) Efficiency",
    width=550, height=300, x_axis_type="linear",
)
peff.add_layout(models.Legend(), 'right')

# Add the new plot to the layout
io.curdoc().add_root(row(peff))

# Call the new plot function
plotGeneratorEfficiency()

def plotGeneratorFrequency():
    global i
    # Create a new ColumnDataSource for generator frequency
    outFreqGens = []
    colors = ["red", "blue", "green", "pink", "brown"]
    for j, gen in enumerate([x for x in mm.keys() if x.startswith('generator')]):
        outFreqGens.append(models.ColumnDataSource(data=dict(Time=[i], Frequency=[0])))
        pfreq.line(x="Time", y="Frequency", source=outFreqGens[j], width=2, color=colors[j], legend_label=f"{gen} Frequency")

# Create a new figure for generator frequency
pfreq = plotting.figure(
    x_axis_label="Time", y_axis_label="Frequency", title="Generator(s) Frequency",
    width=550, height=300, x_axis_type="linear",
)
pfreq.add_layout(models.Legend(), 'right')

# Add the new plot to the layout
io.curdoc().add_root(row(pfreq))

# Call the new plot function
plotGeneratorFrequency()

def plotGeneratorVoltages():
    global i, outVGens
    colors = ["red", "blue", "green", "pink", "brown"]
    for j, gen in enumerate([x for x in mm.keys() if x.startswith('generator')]):
        outVGens.append(models.ColumnDataSource(data=dict(Time=[i], VoltageOutput=[0])))
        pv.line(x="Time", y="VoltageOutput", source=outVGens[j], width=2, color=colors[j], legend_label=gen)

# Create a new figure for generator voltages
pv = plotting.figure(
    x_axis_label="Time", y_axis_label="VoltageOutput", title="Generator(s) Output Voltage (V)",
    width=550, height=300, x_axis_type="linear",
)
pv.add_layout(models.Legend(), 'right')

# Add the new plot to the layout
io.curdoc().add_root(row(pv))

# Call the new plot function
plotGeneratorVoltages()

# Replicate the voltage plot for frequency
def plotGeneratorFrequencies():
    global i, outFreqGens
    colors = ["red", "blue", "green", "pink", "brown"]
    for j, gen in enumerate([x for x in mm.keys() if x.startswith('generator')]):
        outFreqGens.append(models.ColumnDataSource(data=dict(Time=[i], Frequency=[0])))
        pfreq.line(x="Time", y="Frequency", source=outFreqGens[j], width=2, color=colors[j], legend_label=gen)

# Create a new figure for generator frequencies
pfreq = plotting.figure(
    x_axis_label="Time", y_axis_label="Frequency", title="Generator(s) Frequency",
    width=550, height=300, x_axis_type="linear",
)
pfreq.add_layout(models.Legend(), 'right')

# Add the new plot to the layout
io.curdoc().add_root(row(pfreq))

# Call the new plot function
plotGeneratorFrequencies()

In [ ]:
import psutil, os, socket

for interface, addrs in psutil.net_if_addrs().items():
    # Skip Docker-related interfaces
    if interface.startswith("docker") or interface.startswith("br-"):
        continue

    for addr in addrs:
        if addr.family == socket.AF_INET and not addr.address.startswith("127."):
            print(f"Interface: {interface}, LAN IP: {addr.address}")
            os.environ["IP"] = addr.address
            break

In [ ]:
!python3 -m bokeh  serve streaming --allow-websocket-origin='*' --address $IP --port 5006
